# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze a complex multi-record-set dataset defined by a Croissant schema, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and records from the FAIR$^2$ dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview

Let's list available record sets, their `@id` fields, as well as fields/columns inside each record set. All references to elements in the data will be made **via their `@id` fields**.

We use the Croissant library to access the record sets. Please note that each record set, field, and column has a unique `@id` within the schema, which will be displayed.

In [ ]:
# List all available record sets and their fields by @id
record_sets_info = dataset.list_record_sets()
all_recordset_ids = []
for rs in record_sets_info:
    print(f"RecordSet @id: {rs['@id']}")
    all_recordset_ids.append(rs['@id'])
    if 'fields' in rs and rs['fields']:
        for fld in rs['fields']:
            print(f"  Field/Column @id: {fld['@id']} | name: {fld.get('name', '-')}")
    print("--")

# Optionally, display the full list of record set ids
print(f"\nRecord set IDs: {all_recordset_ids}")

## 3. Data Extraction

Now, let's load records from one or more record sets into Pandas DataFrames. 

**All access to record sets and fields is via their `@id`.**

Below, we extract the full list of record set `@id`s discovered above, and load their records as separate DataFrames.

In [ ]:
dataframes = {}
# Use the record set @ids discovered in section 2
for record_set_id in all_recordset_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for RecordSet @id: {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(2), "\n")
    except Exception as e:
        print(f"Could not load records for RecordSet @id: {record_set_id}. Error: {e}")

# For the next steps, select one main record set, e.g., the first one.
if all_recordset_ids:
    main_record_set_id = all_recordset_ids[0]
    print(f"Selected main RecordSet for analysis: {main_record_set_id}")
    print("Available columns:", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Now, let's perform some EDA: filter on a numeric field, normalize it, and (optionally) group data by a categorical field. 

All field references will use their full `@id`, which can be found in the previous sections' output. Please adjust the field ids below if your dataset uses different names.

In [ ]:
# Example: Use the first available numeric-looking column
df = dataframes[main_record_set_id]

# Identify numeric fields by dtype
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print("Numeric fields detected:", numeric_fields)

if numeric_fields:
    numeric_field_id = numeric_fields[0]  # Could be e.g. 'dv:log_likelihood' or similar id
    print(f"Using numeric field: {numeric_field_id}")
    threshold = df[numeric_field_id].mean() + df[numeric_field_id].std()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Now try grouping by a likely categorical field
    # For demonstration, use the first object or category dtype column that isn't numeric.
    group_fields = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id]
    if group_fields:
        group_field_id = group_fields[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
        print(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization

Let's visualize the distribution of a numeric field, and (optionally) the group-wise mean using Matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    # If we have grouped_df from above
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(10,4))
        sns.barplot(data=grouped_df, x=grouped_df.columns[0], y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {grouped_df.columns[0]}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

- We've used the `mlcroissant` library to programmatically load the metadata and data of a FAIR$^2$ Croissant dataset via its schema URL.
- All dataset components—record sets, fields, columns—were referenced strictly by their Croissant `@id` identifiers, ensuring compliant, reproducible data access.
- Basic EDA and visualization show how to filter, normalize, group, and plot data from the structured Croissant tables.
- This approach can be adapted for other Croissant-compliant, multi-table, or multi-modal datasets to enable robust, standards-compliant data science workflows.